# Data exploration

Subset of Pl@ntNet-300K with our 20 species. Run `00_download_data.ipynb` first (only once, the data is already in the shared Drive folder).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy the zip from Drive and unzip it on the Colab disk (1-2 min)
!cp /content/drive/MyDrive/plantnet_dlnn26/plantnet20_v1.zip /content/
!unzip -q -o /content/plantnet20_v1.zip -d /content/plantnet20

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

drive_dir = '/content/drive/MyDrive/plantnet_dlnn26'
data_dir = '/content/plantnet20'

# 20 species, in pairs of the same genus (same order as the README)
species_ids = [4, 3, 112, 225, 84, 171, 200, 202, 190, 15, 11, 406, 47, 46, 60, 186, 35, 243, 310, 366]

sel = pd.read_csv(drive_dir + '/selection.csv')
sel['file'] = data_dir + '/' + sel['path']
print(sel.shape)
sel.head()

## Images per species and split

In [ ]:
counts = sel.pivot_table(index='species', columns='split', values='file', aggfunc='count')
counts = counts[['train', 'val', 'test']]
# keep the pair order
names = sel.drop_duplicates('species_id').set_index('species_id')['species']
counts = counts.loc[[names[s] for s in species_ids]]
counts

In [ ]:
counts.plot(kind='barh', stacked=True, figsize=(8, 8))
plt.gca().invert_yaxis()
plt.xlabel('images')
plt.title('Images per species')
plt.show()

# train is capped at 1000, but val and test are not balanced
print(counts.describe())

## Organs

In [ ]:
organs = pd.crosstab(sel['species'], sel['organ'], normalize='index').round(2)
organs = organs.loc[counts.index]
organs

In [ ]:
# Overall, by split
pd.crosstab(sel['organ'], sel['split'])

## Image sizes

On a random sample, to choose the input size of the network.

In [ ]:
sample = sel.sample(500, random_state=42)
sizes = []
for f in sample['file']:
    with Image.open(f) as img:
        sizes.append(img.size)
sizes = pd.DataFrame(sizes, columns=['width', 'height'])
print(sizes.describe())

plt.scatter(sizes['width'], sizes['height'], alpha=0.3)
plt.xlabel('width')
plt.ylabel('height')
plt.title('Image sizes (sample of 500)')
plt.show()

## Some examples

Two train images per species. Each row is one pair of the same genus.

In [ ]:
train = sel[sel['split'] == 'train']

fig, axes = plt.subplots(10, 4, figsize=(12, 30))
for i in range(10):
    for j in range(2):
        sp = species_ids[2 * i + j]
        imgs = train[train['species_id'] == sp].sample(2, random_state=1)
        for k in range(2):
            ax = axes[i, 2 * j + k]
            ax.imshow(Image.open(imgs['file'].iloc[k]))
            ax.set_title(names[sp] + '\n' + imgs['organ'].iloc[k], fontsize=9)
            ax.axis('off')
plt.tight_layout()
plt.show()

## Same author in train and test

If the same person took photos of the same species in train and test, the photos can be very similar (same plant, same place). We check how much of the test set this is.

In [ ]:
train_pairs = set(zip(train['species_id'], train['author']))
test = sel[sel['split'] == 'test'].copy()
test['author_in_train'] = [(s, a) in train_pairs for s, a in zip(test['species_id'], test['author'])]

print('test images with an author also in train:', round(test['author_in_train'].mean() * 100, 1), '%')
test.groupby('species')['author_in_train'].mean().sort_values(ascending=False).round(2)